## EDA --- FOR POSSIBLE FEAT TRANSFORM

# imports and loading

In [9]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

train_path = ROOT / "data" / "processed" / "train.csv"
test_path = ROOT / "data" / "processed" / "test.csv"

print("Project folder:", ROOT)
print("Training file found:", train_path.exists())
print("Test file found:", test_path.exists())

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training shape:", train.shape)
print("Test shape:", test.shape)

Project folder: c:\Users\anton\Desktop\Mestrado\Group_20_ETAI
Training file found: True
Test file found: True
Training shape: (6469, 39)
Test shape: (986, 38)


# feature basic info

In [10]:
print("Training columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nTraining data types:")
display(train.dtypes.to_frame("data_type"))

Training columns:
['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'repeat_purchase_90d', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_total_spend', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d', 'customer_id', 'payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'purchase_date', 'ID', 'random_noise']

Test columns:
['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'days_since_previous_pu

,data_type
Total a Pagar,float64
Total Bruto,float64
Total Liquido,float64
Total Imp_,float64
Total Portes,float64
Total Desc_ Global,float64
Total Desc_ Linha,float64
Custo,float64
N_ Detalhes,float64
invoice_count_current_purchase,float64


In [11]:
expected_targets = ["repeat_purchase_90d", "spend_next_90d"]

for target in expected_targets:
    print(f"{target}: {'present' if target in train.columns else 'missing'}")

repeat_purchase_90d: present
spend_next_90d: missing


# purchase date range

In [ ]:
train_dates = pd.to_datetime(train["purchase_date"])
test_dates = pd.to_datetime(test["purchase_date"])

print("Train:", train_dates.min().date(), "to", train_dates.max().date())
print("Test:", test_dates.min().date(), "to", test_dates.max().date())
print("Days between train and test:", (test_dates.min() - train_dates.max()).days)

# target summary

In [ ]:
print(train["repeat_purchase_90d"].value_counts())

returned = train[train["repeat_purchase_90d"] == 1]
print("Missing spend among repeat purchases:", returned["spend_next_90d"].isna().sum())

# missing spend by year

In [ ]:
missing_spend = returned["spend_next_90d"].isna()
by_year = missing_spend.groupby(train_dates.dt.year).agg(repeat_purchases="size", missing_spend="sum")
by_year["missing_pct"] = (100 * by_year["missing_spend"] / by_year["repeat_purchases"]).round(1)
display(by_year)

# missing feature values

In [7]:
features = test.columns.drop(["ID", "customer_id"])
missing_pct = pd.DataFrame({
    "train_pct": train[features].isna().mean() * 100,
    "test_pct": test[features].isna().mean() * 100,
})
display(missing_pct.nlargest(10, ["train_pct", "test_pct"]).round(1))

no_history = train["prior_purchase_count"].eq(0).rename("no_prior_purchases")
display(pd.crosstab(no_history, train["days_since_previous_purchase"].isna(),
                    colnames=["recency_missing"]))

                              train_pct  test_pct
prior_std_gap_days                 21.8       7.0
prior_mean_gap_days                19.2       7.7
prior_std_purchase_value           16.2       4.7
prior_mean_purchase_value          12.3       5.3
days_since_previous_purchase       12.3       5.3
prior_min_purchase_value            9.3       2.2
prior_max_purchase_value            9.3       2.2
transport_method                    3.0       3.0
spend_365d                          3.0       3.0
payment_type                        3.0       3.0
recency_missing     False  True 
no_prior_purchases              
False                5673    193
True                    0    603


# repeat purchase rate by recency

In [8]:
recency = pd.cut(train["days_since_previous_purchase"],
                 bins=[0, 30, 90, float("inf")], labels=["0–30", "31–90", "91+"],
                 include_lowest=True)
recency = recency.cat.add_categories(["Missing", "Out of range"])
recency = recency.mask(train["days_since_previous_purchase"].isna(), "Missing")
recency = recency.fillna("Out of range")
recency_summary = train.groupby(recency, observed=True)["repeat_purchase_90d"]
display(recency_summary.agg(purchases="size", repeat_rate="mean")
        .assign(repeat_pct=lambda summary: (100 * summary.pop("repeat_rate")).round(1)))

                              purchases  repeat_pct
days_since_previous_purchase                       
0–30                               2057        72.6
31–90                              1396        67.8
91+                                2204        39.2
Missing                             796        39.1
Out of range                         16        31.2


# spend target distribution

In [9]:
display(returned["spend_next_90d"].describe(percentiles=[0.5, 0.95]).round(2))

count     3425.00
mean      1921.53
std       2571.05
min          1.27
50%        985.74
95%       6800.26
max      25654.33
Name: spend_next_90d, dtype: float64


# temporal validation split

In [10]:
validation_start = train_dates.sort_values().iloc[int(len(train) * 0.8)]
validation_rows = train_dates >= validation_start
training_rows = train_dates + pd.Timedelta(days=90) < validation_start
gap_rows = ~(training_rows | validation_rows)
has_spend_target = train["repeat_purchase_90d"].eq(1) & train["spend_next_90d"].notna()

print("Validation starts:", validation_start.date())
print("Rows excluded for the 90-day target window:", gap_rows.sum())
display(pd.DataFrame({
    "rows": [training_rows.sum(), validation_rows.sum()],
    "observed_spend_targets": [(training_rows & has_spend_target).sum(),
                               (validation_rows & has_spend_target).sum()],
}, index=["Train", "Validation"]))

Validation starts: 2023-06-17
Rows excluded for the 90-day target window: 147
            rows  observed_spend_targets
Train       5024                    2656
Validation  1298                     694


# negative values and transaction totals

In [11]:
amounts = ["Total a Pagar", "Total Bruto", "Total Liquido"]
checked = amounts + ["days_since_previous_purchase"]
display(pd.DataFrame({"train_negative": train[checked].lt(0).sum(),
                      "test_negative": test[checked].lt(0).sum()}))
display(train[amounts].describe(percentiles=[0.99]).loc[["min", "50%", "99%", "max"]].round(2))

                              train_negative  test_negative
Total a Pagar                              0              0
Total Bruto                                0              0
Total Liquido                              0              0
days_since_previous_purchase              16              2
     Total a Pagar  Total Bruto  Total Liquido
min           0.93         0.83           0.76
50%         476.74       445.74         389.91
99%        6543.97      6717.62        5675.70
max       22297.21     21578.28       18127.81


# validation profile

In [12]:
fit_rows = train.loc[training_rows]
validation = train.loc[validation_rows]
for name, split in [("Train", fit_rows), ("Validation", validation)]:
    repeats = split[split["repeat_purchase_90d"].eq(1)]
    print(name, "| Repeat purchase (%):", round(100 * len(repeats) / len(split), 1),
          "| Missing spend among repeats (%):", round(100 * repeats["spend_next_90d"].isna().mean(), 1))

unseen = ~validation["customer_id"].isin(fit_rows["customer_id"])
print("Unseen validation customers:", validation.loc[unseen, "customer_id"].nunique(),
      "of", validation["customer_id"].nunique())

Train | Repeat purchase (%): 55.0 | Missing spend among repeats (%): 3.8
Validation | Repeat purchase (%): 60.3 | Missing spend among repeats (%): 11.4
Unseen validation customers: 47 of 241
